# AXIOM · Codex × GPT-5.6 — Demo Notebook

> A walkthrough of what I built for **DSH Hacks V1**, and exactly how two frontier models — **OpenAI Codex (cloud coding agent)** and **GPT-5.6** — were used to build it.

**Live demo**: https://axiom-stem.pages.dev
**Repo**: https://github.com/radhikatmosphere/axiom-stem
**Hackathon**: https://dsh-hacks-v1.devpost.com/
**Author**: [RADHIKATMOSPHERE](https://github.com/radhikatmosphere)

---

### At a glance

| What | Where it lives | Who wrote it |
|---|---|---|
| Deterministic Layer 1 engine (genetics, combinatorics, chemistry, physics) | `lib/decomposers.ts` | **Codex** |
| Test harness (esbuild bundle, 13 assertions) | `tests/decomposers.test.mjs` | **Codex** |
| Bench probe scripts | `bench/` | **Codex** |
| Repo conventions (AGENTS.md) | `AGENTS.md` | **Codex** |
| Narrative Layer 2 copy, Devpost text, pitch deck | `submission/*.md`, README prose | **GPT-5.6** |
| Architecture framing (Chandas ↔ AXIOM) | README §"Chandas Inspiration" | **GPT-5.6** |
| This notebook + the Devpost demo video script | `notebooks/`, `submission/DEMO_GUION.md` | **GPT-5.6** |

Both outputs are human-reviewed and committed through version control.



## 1 · The problem AXIOM solves

Most AI tutors *guess* their way through STEM math. Students get confident-sounding explanations that are subtly wrong — especially in:

- Genotype probabilities (a regular LLM will happily say `Aa × aa → 50% AA, 50% aa`, which is wrong)
- Combinatorics boundary cases (`C(n,0)`, repeated-`r` variants)
- Electron configurations (noble-gas shortcuts, valence electrons for transition metals)
- Harmonic frequencies (octave doubling, partial wavelengths)

**AXIOM's bet**: split the work in two. A *deterministic* Layer 1 engine computes the exact answer — no network, no hallucination. A *narrative* Layer 2 turns that exact JSON into a vivid, age-appropriate explanation. *Compute first, explain second.*



## 2 · The dual-engine architecture

```
Layer 1 — Combinatorial Decomposer (TypeScript, pure, instant)
    ↓ exact JSON
Layer 2 — Narrative Adapter (xAI Grok / SuperGrok)
    ↓ vivid, age-appropriate (13–18)
Student
```

**Layer 1 is the only thing that touches numbers.** It's about 350 lines of typed TS, four domains, zero external calls. Codex wrote it.



## 3 · The Chandas inspiration (framing by GPT-5.6)

This whole paradigm of *decompose → enumerate → narrate* is 2,200 years old. It comes from **Sanskrit prosody** — specifically the **Chandas** tradition credited to **Piṅgala**.

Piṅgala's *prastāra* operation enumerated every valid rhythmic pattern for a stanza by:

1. Listing the atomic units (the *gaṇa* feet — Laghu / Guru)
2. Enumerating every legal combination row by row
3. Marking the boundaries (*yati* caesuras)
4. Only *then* commenting on the meaning

AXIOM does the same thing:

| Chandas | AXIOM Layer 1 |
|---|---|
| Stanza / pādas | STEM domains (genetics, math, chemistry, physics) |
| Gaṇa (L/G syllables) | Combinatorial atoms (alleles, C(n,r) steps) |
| Yati (caesura) | Step boundaries in output |
| Prastāra | Punnett grids, formula expansion, Aufbau walk |

Putting Piṅgala next to AXIOM wasn't an accident — it's the entire intellectual lineage. GPT-5.6 drafted this table and the surrounding prose; Codex had no opinion about Sanskrit prosody, and that's the point.



## 4 · Layer 1 — a deterministic TS engine (built by Codex)

Below is a faithful Python *re-implementation* of Layer 1's core algorithms. It mirrors what Codex wrote in `lib/decomposers.ts` so you can run it inside Colab without spinning up Node.

This cell is the only code that's doing real work. The rest of the notebook explains *why* it exists, *who* built it, and *how* to deploy it.


In [ ]:
from collections import Counter
from functools import reduce
from operator import mul
from math import factorial

ORBITAL_ORDER = ["1s", "2s", "2p", "3s", "3p", "4s", "3d", "4p", "5s", "4d", "5p", "6s", "4f", "5d"]
CAPACITY = {"s": 2, "p": 6, "d": 10, "f": 14}
SUPER = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")
NOBLES = [
    ("",  0),
    ("He", 2, "1s"),
    ("Ne", 10, "2p"),
    ("Ar", 18, "3p"),
    ("Kr", 36, "4p"),
]

# ── GENETICS (Codex) ──────────────────────────────────────────────────────
def parse_genotype(g):
    g = g.strip()
    if len(g) != 2 or not g[0].isalpha() or not g[1].isalpha():
        raise ValueError("Genotype must be 2 letter alleles (e.g. Aa, BB)")
    if g[0].upper() != g[1].upper():
        raise ValueError("Both alleles must describe the same gene")
    return g[0], g[1]

def decompose_genetics(p1, p2):
    a1, b1 = parse_genotype(p1)
    a2, b2 = parse_genotype(p2)
    grid = []
    counts = Counter()
    for g1 in (a1, b1):
        row = []
        for g2 in (a2, b2):
            o = "".join(sorted((g1, g2)))
            row.append(o)
            counts[o] += 1
        grid.append(row)
    total = sum(counts.values())
    probs = {k: round(v / total * 100, 2) for k, v in counts.items()}
    return {"punnett": grid, "genotypeProbabilities": probs}

# ── MATH (Codex) ──────────────────────────────────────────────────────────
def nCr(n, r):
    if r < 0 or r > n: return 0
    return factorial(n) // (factorial(r) * factorial(n - r))
def nPr(n, r):
    if r < 0 or r > n: return 0
    return factorial(n) // factorial(n - r)

# ── CHEMISTRY (Codex, fixed for transition-metal noble-gas cut-over) ────
def superscript(n):
    return str(n).translate(SUPER)

def electron_config(Z):
    remaining = Z
    orbitals = {}
    steps = []
    for orb in ORBITAL_ORDER:
        if remaining <= 0: break
        cap = CAPACITY[orb[-1]]
        fill = min(remaining, cap)
        orbitals[orb] = fill
        remaining -= fill
        steps.append(f"Fill {orb}: {fill} electrons (max {cap})")
    full = " ".join(f"{o}{superscript(c)}" for o, c in orbitals.items())
    return full, steps, orbitals

def noble_gas_config(Z, orbitals):
    noble = NOBLES[0]
    for ng in NOBLES[1:]:
        if ng[1] < Z and ng[1] > noble[1]:
            noble = ng
    if noble[1] == 0:
        return electron_config(Z)[0]
    noble_idx = ORBITAL_ORDER.index(noble[2]) if noble[2] else -1
    tail = " ".join(
        f"{o}{superscript(c)}" for o, c in orbitals.items()
        if ORBITAL_ORDER.index(o) > noble_idx
    )
    return f"[{noble[0]}] {tail}".strip()

def valence_electrons(orbitals, Z):
    if not orbitals: return 0
    max_shell = max(int(o[0]) for o in orbitals)
    v = 0
    for o, c in orbitals.items():
        shell = int(o[0])
        if shell == max_shell or (shell == max_shell - 1 and o.endswith("d")):
            v += c
    return v

# ── PHYSICS (Codex) ───────────────────────────────────────────────────────
def harmonics(f0, n):
    out = []
    speed_of_sound = 343.0
    for i in range(1, n + 1):
        f = f0 * i
        out.append({"n": i, "frequency": f, "wavelength": round(speed_of_sound / f, 4)})
    return out

print("AXIOM Layer 1 (Python port, faithful to Codex's lib/decomposers.ts) loaded ✓")


## 5 · Try the engine — three live tiles

These cells run Layer 1 against the canonical demo inputs from the live site. Every result is exact — no probabilistic sampling, no network call.


In [ ]:
# Genetics: Aa × aa
g = decompose_genetics("Aa", "aa")
print("Aa × aa → Punnett grid:", g["punnett"])
print("Genotype probabilities:", g["genotypeProbabilities"])
# Expected: Aa=50%, aa=50%  (NOT 50/50 AA/aa — that's the common LLM mistake)


In [ ]:
# Math: combination and permutation baseline
print(f"C(5,3)  = {nCr(5, 3)}      (expected 10)")
print(f"C(10,4) = {nCr(10, 4)}     (expected 210)")
print(f"P(5,3)  = {nPr(5, 3)}      (expected 60)")
print(f"P(7,0)  = {nPr(7, 0)}      (expected 1)")


In [ ]:
# Chemistry: iron and sodium
full, steps, orbs = electron_config(26)
print("Fe (Z=26) build steps:")
for s in steps: print("  -", s)
print()
print("Fe full config:    ", full)
print("Fe noble-gas:      ", noble_gas_config(26, orbs))
print("Fe valence:        ", valence_electrons(orbs, 26))
# Expected: [Ar] 4s² 3d⁶, valence 8

full, steps, orbs = electron_config(11)
print()
print("Na (Z=11) noble-gas:", noble_gas_config(11, orbs))
print("Na valence:         ", valence_electrons(orbs, 11))


In [ ]:
# Physics: harmonic series on A = 440 Hz, eight partials
hs = harmonics(440, 8)
for h in hs:
    print(f"  n={h['n']}  f={h['frequency']:>5} Hz   λ={h['wavelength']} m")


## 6 · How I used OpenAI Codex

Codex was the **primary implementation agent** for the deterministic core of AXIOM and for everything that touches the build system.

### 6.1 The Codex session for this submission

The active Codex session that landed the finalization pass for this hackathon submission is:

```
session id: 019f7d24-6a15-7f02-8eee-67b72267dfe2
timestamp:   2026-07-19T19:29:26 (local)
artifacts:   ~/.codex/sessions/2026/07/19/rollout-2026-07-19T19-29-26-*.jsonl
```

Three earlier Codex sessions on the same day scaffolded the project and did earlier iteration:

```
019f7cef-7eea-75f0-a8aa-272f43850578   18:31 — initial scaffold + repo conventions
019f7cf0-44fd-7743-8c46-8c36a97cd6a6   18:32 — Layer 1 stub
019f7cf1-0dad-7523-bc67-1604f9be196b   18:33 — Layer 1 first full pass
019f7d24-6a15-7f02-8eee-67b72267dfe2   19:29 — finalization for this submission
```

(All four are persisted under `~/.codex/sessions/2026/07/19/`.)

### 6.2 What Codex actually did, file by file

| Codex output | Where it lives | Verification |
|---|---|---|
| **`lib/decomposers.ts`** — Punnett grid, `C(n,r)` / `P(n,r)`, Aufbau, harmonic series | `lib/decomposers.ts` | esbuild bundle compiles cleanly, 13 assertions green |
| **`tests/decomposers.test.mjs`** — esbuild-bundled Node test runner | `tests/decomposers.test.mjs` | `node tests/decomposers.test.mjs` → 0 exit |
| **`bench/probe.mjs` … `probe4.mjs`** — perf probes for the Layer 1 functions | `bench/` | runnable with `node bench/probe*.mjs` |
| **`AGENTS.md`** — repo conventions file | `AGENTS.md` | human-readable, used by Codex and humans equally |

### 6.3 The interesting failure mode Codex surfaced

While finalizing for this submission Codex briefly introduced a corrupted state in `lib/decomposers.ts` (a stray NUL byte in the genetics-validation regex, plus a truncated noble-gas-config helper). The bundle errored and the chemistry test failed. Codex's own re-read caught it; for this submission I:

- Stripped the stray NUL byte and closed the regex (`/^[A-Za-z]{2}$/.test(g)`)
- Added a typed `lastOrbital` field to the noble-gases table so the noble-gas cut-over is no longer parsed out of unicode-superscripted config strings
- Re-ran `node tests/decomposers.test.mjs` → all 13 PASS

That last bullet is the whole point of having deterministic Layer 1: **Codex's mistakes in code are caught by code, not by humans at the demo.**



## 7 · How I used GPT-5.6

GPT-5.6 owned everything that has to *read well*: narrative copy on the live tiles, the Devpost/Discord/pitch submission text, the architecture story, and this notebook.

### 7.1 What GPT-5.6 actually wrote

| GPT-5.6 output | Where it lives | Notes |
|---|---|---|
| **"AI Tooling — How Codex & GPT-5.6 Were Used"** section | `README.md` | this same division-of-labor you are reading in this notebook, mirrored into the README for the devpost submission |
| **Chandas ↔ AXIOM table and prose** | `README.md` §"Chandas Inspiration" | the connection to Piṅgala's *prastāra* was GPT-5.6's framing |
| **Layer 2 explanation copy** | `app/` (tile copy chosen at runtime based on Layer 1 JSON) | the friendly "Aa is like a coin flip per allele" voice |
| **`submission/DEVPOST_COPYPASTE.md`** | `submission/` | pre-filled hackathon fields |
| **`submission/PROJECT_DESCRIPTION.md`** | `submission/` | one-pager |
| **`submission/AXIOM_AI_Pitch_Deck.md`** | `submission/` | pitch outline |
| **`submission/DEMO_VIDEO_SCRIPT.md` & `DEMO_GUION.md`** | `submission/` | the demo video timeline (script + Spanish *guión*) |
| **This notebook** | `notebooks/AXIOM_Demo_Codex_GPT56.ipynb` | walks through what was built and who built what |

### 7.2 Why split this way

Codex writes code. GPT-5.6 writes prose. They overlap very little.

- Codex is at its best with **typed, deterministic targets** (regex, atomic loops, JSON shapes).
- GPT-5.6 is at its best with **narrative, framing, and structured prose** (docs, decks, demo scripts).

Mixing the two — asking Codex to write poetry, or GPT-5.6 to typecheck C-numerators — would have wasted both. So the rule I followed was:

> **If the output has to compile, Codex. If it has to *persuade*, GPT-5.6.**



## 8 · Verification & deploy (what I'd run on a clean machine)

```bash
git clone https://github.com/radhikatmosphere/axiom-stem.git
cd axiom-stem
npm install
node tests/decomposers.test.mjs      # → all 13 PASS, exit 0
npm run lint                          # → next/eslint clean
npm run dev                           # localhost:3000
npm run pages:deploy                  # ship to Cloudflare Pages
```

Production is live at **<https://axiom-stem.pages.dev>**.

Telemetry (decompose, narrative_generated, auth_connect, error) ships to Splunk index **`axiom`**. Full SPL dashboards live at `docs/SPLUNK_DASHBOARDS.md`.


## 9 · Closing — the rule AXIOM enforces on AI

> **Compute first. Explain second.**

Whether the AI writing your chemistry answer is Codex, GPT-5.6, or Grok, the deterministic engine underneath does not lie. The narrative layer narrates a *result*, not *an opinion shaped to sound like a result*.

*Thank you.*


---

### Credits

- **AXIOM Layer 1**: OpenAI Codex · sessions `019f7d24-…`, `019f7cef-…`, `019f7cf0-…`, `019f7cf1-…`
- **AXIOM Layer 2 narrative + submission prose + this notebook**: GPT-5.6
- **Auth + wallet karma + telemetry + deployment**: xAI SuperGrok, Firebase, Splunk HEC, Cloudflare Pages

Built for **DSH Hacks V1** · AI × STEM Education · MIT licensed

